##### Sequence-level mixed-state representation

For each modality, it uses the real hidden-state sequence `[T, 768]`, projects it to `r=16`, and constructs an attention-weighted mixed density state:

\[
\rho_m = \sum_t \alpha_t |\psi_t\rangle\langle\psi_t|.
\]

The tensor-product joint state `rho_a ⊗ rho_t` is included only as a sanity-check state. It is **not** the final correlated interaction state.

### Important

For the product state, QMI is zero by construction. The final proposed interaction must create a post-interaction joint state before QMI is used as a model feature.

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OFFICIAL_DIR = PROCESSED_DIR / "official_split"
QUANTUM_DIR = PROCESSED_DIR / "quantum"

# Stage C sequence-level outputs
TRAIN_PKL = QUANTUM_DIR / "train.pkl"
DEV_PKL = QUANTUM_DIR / "dev.pkl"
TEST_PKL = QUANTUM_DIR / "test.pkl"

RANK = 16
EPS = 1e-8

print("Project root:", PROJECT_ROOT)
print("Quantum sequence data:", QUANTUM_DIR)
print("Reduced rank:", RANK)


## Sequence-input validation

The corrected quantum representation **requires hidden-state sequences**, not pooled 768-D vectors.

In [ ]:
def load_pickle(path):
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    with open(path, "rb") as f:
        return pickle.load(f)

AUDIO_SEQUENCE_KEYS = [
    "audio_hidden_states", "audio_hidden_sequence",
    "hubert_hidden_states", "hubert_hidden_sequence",
]
TEXT_SEQUENCE_KEYS = [
    "text_hidden_states", "text_hidden_sequence",
    "deberta_hidden_states", "deberta_hidden_sequence",
]

def find_first_key(record, candidates):
    for key in candidates:
        if key in record:
            return key
    return None

def describe_sequence_inputs(records, n=3):
    for i, record in enumerate(records[:n]):
        akey = find_first_key(record, AUDIO_SEQUENCE_KEYS)
        tkey = find_first_key(record, TEXT_SEQUENCE_KEYS)
        print(f"Record {i}: participant={record.get('participant_id')}, "
              f"utterance={record.get('utterance_id')}")
        print("  audio:", akey, np.asarray(record[akey]).shape if akey else None)
        print("  text :", tkey, np.asarray(record[tkey]).shape if tkey else None)

train_records = load_pickle(TRAIN_PKL)
dev_records = load_pickle(DEV_PKL)
test_records = load_pickle(TEST_PKL)

print("Loaded records:")
print("  train:", len(train_records))
print("  dev  :", len(dev_records))
print("  test :", len(test_records))

for name, records in [("train", train_records), ("dev", dev_records), ("test", test_records)]:
    participants = {str(r["participant_id"]) for r in records}
    print(f"  {name}: {len(participants)} participants")

describe_sequence_inputs(train_records)


In [10]:
def validate_sequence_records(records, split_name):
    if not records:
        raise ValueError(f"No records loaded for {split_name}.")

    missing_audio, missing_text = [], []
    bad_audio, bad_text = [], []
    nonfinite_audio, nonfinite_text = [], []

    for i, record in enumerate(records):
        akey = find_first_key(record, AUDIO_SEQUENCE_KEYS)
        tkey = find_first_key(record, TEXT_SEQUENCE_KEYS)

        if akey is None:
            missing_audio.append(i)
        else:
            a = np.asarray(record[akey])
            if a.ndim != 2 or a.shape[1] != 768 or a.shape[0] < 1:
                bad_audio.append((i, a.shape))
            elif not np.isfinite(a).all():
                nonfinite_audio.append(i)

        if tkey is None:
            missing_text.append(i)
        else:
            t = np.asarray(record[tkey])
            if t.ndim != 2 or t.shape[1] != 768 or t.shape[0] < 1:
                bad_text.append((i, t.shape))
            elif not np.isfinite(t).all():
                nonfinite_text.append(i)

    participants = {str(r["participant_id"]) for r in records}
    print(f"{split_name}: {len(records)} records, {len(participants)} participants")
    print("  missing audio:", len(missing_audio))
    print("  missing text :", len(missing_text))
    print("  bad audio    :", bad_audio[:5])
    print("  bad text     :", bad_text[:5])
    print("  nonfinite audio/text:", len(nonfinite_audio), len(nonfinite_text))

    if missing_audio or missing_text or bad_audio or bad_text or nonfinite_audio or nonfinite_text:
        raise ValueError(f"{split_name} failed sequence validation.")

for name, records in [("train", train_records), ("dev", dev_records), ("test", test_records)]:
    validate_sequence_records(records, name)

train_participants = {str(r["participant_id"]) for r in train_records}
dev_participants = {str(r["participant_id"]) for r in dev_records}
test_participants = {str(r["participant_id"]) for r in test_records}

assert not (train_participants & dev_participants)
assert not (train_participants & test_participants)
assert not (dev_participants & test_participants)
assert len(train_participants) == 163
assert len(dev_participants) == 56
assert len(test_participants) == 56

print("Official participant split integrity: PASS")

train: 13451 records, 163 participants
  missing audio: 0
  missing text : 0
  bad audio    : []
  bad text     : []
  nonfinite audio/text: 0 0
dev: 4904 records, 56 participants
  missing audio: 0
  missing text : 0
  bad audio    : []
  bad text     : []
  nonfinite audio/text: 0 0
test: 5081 records, 56 participants
  missing audio: 0
  missing text : 0
  bad audio    : []
  bad text     : []
  nonfinite audio/text: 0 0
Official participant split integrity: PASS


## Sequence projection

Each frame/token vector is projected from 768 dimensions to a shared reduced dimension \(r=16\).

A shared projection is used so that the speech and text density states live in the same reduced Hilbert space.

For each modality:

\[
h_k = W x_k
\]

followed by normalization:

\[
\psi_k = \frac{h_k}{\|h_k\|_2}.
\]

In [11]:
class SequenceProjector(torch.nn.Module):
    def __init__(self, input_dim=768, rank=16):
        super().__init__()
        self.proj = torch.nn.Linear(input_dim, rank, bias=False)

    def forward(self, x):
        # x: [T, 768]
        h = self.proj(x)
        h = F.normalize(h, p=2, dim=-1, eps=EPS)
        return h

projector = SequenceProjector(input_dim=768, rank=RANK)

print(projector)


SequenceProjector(
  (proj): Linear(in_features=768, out_features=16, bias=False)
)


## Attention-weighted mixed density state

Instead of one outer product, the density state is a weighted mixture over the sequence:

\[
\rho_m =
\sum_{k=1}^{T_m}
\alpha_k \,
\psi_k \psi_k^{\top}
\]

where

\[
\alpha_k \ge 0,
\qquad
\sum_k \alpha_k = 1.
\]

This allows the state to have rank greater than one and therefore avoids the previous pure-state degeneracy.

The attention weights are generated from the projected sequence.

In [12]:
class AttentionDensity(torch.nn.Module):
    def __init__(self, rank=16):
        super().__init__()
        self.score = torch.nn.Linear(rank, 1, bias=False)

    def forward(self, states):
        # states: [T, rank], assumed L2-normalized
        logits = self.score(states).squeeze(-1)
        alpha = torch.softmax(logits, dim=0)

        rho = torch.einsum(
            "t,ti,tj->ij",
            alpha,
            states,
            states
        )

        # Numerical trace normalization
        rho = rho / rho.trace().clamp_min(EPS)

        return rho, alpha

density_head = AttentionDensity(rank=RANK)
print(density_head)


AttentionDensity(
  (score): Linear(in_features=16, out_features=1, bias=False)
)


## Quantum metrics

For a density matrix \(\rho\):

- purity: \(P(\rho)=\mathrm{Tr}(\rho^2)\)
- von Neumann entropy: \(S(\rho)=-\mathrm{Tr}(\rho\log\rho)\)
- trace distance:
  \[
  D(\rho,\sigma)=\frac12\|\rho-\sigma\|_1
  \]

The implementation symmetrises the state and clamps eigenvalues for numerical stability.

In [13]:
def symmetrize_density(rho):
    rho = 0.5 * (rho + rho.T)
    rho = rho / rho.trace().clamp_min(EPS)
    return rho

def purity(rho):
    rho = symmetrize_density(rho)
    return torch.trace(rho @ rho).real

def von_neumann_entropy(rho):
    rho = symmetrize_density(rho)
    eigvals = torch.linalg.eigvalsh(rho).real
    eigvals = torch.clamp(eigvals, min=EPS)
    return -(eigvals * torch.log(eigvals)).sum()

def trace_distance(rho, sigma):
    rho = symmetrize_density(rho)
    sigma = symmetrize_density(sigma)
    delta = 0.5 * ((rho - sigma) + (rho - sigma).T)
    eigvals = torch.linalg.eigvalsh(delta).real
    return 0.5 * eigvals.abs().sum()

def fidelity_pure_to_mixed(state, rho):
    # For normalized real state |psi>, <psi|rho|psi>.
    state = F.normalize(state, p=2, dim=-1, eps=EPS)
    rho = symmetrize_density(rho)
    return torch.real(state @ rho @ state)

def density_frobenius_similarity(rho, sigma):
    rho = symmetrize_density(rho)
    sigma = symmetrize_density(sigma)
    numerator = torch.trace(rho @ sigma).real
    denominator = torch.sqrt(
        torch.trace(rho @ rho).real *
        torch.trace(sigma @ sigma).real
    ).clamp_min(EPS)
    return numerator / denominator


## Joint state and mutual information

The reduced speech and text states are first combined as a tensor-product sanity-check state:

\[
\rho_{at}^{prod} = \rho_a \otimes \rho_t.
\]

For a product state,

\[
I(A:T)=S(\rho_a)+S(\rho_t)-S(\rho_a\otimes\rho_t)=0
\]

up to numerical precision.

Therefore, a zero `product_joint_qmi` in this notebook is **expected**. It is not the final interaction QMI and must not be interpreted as evidence that the proposed model contains no cross-modal information.

The final QMI feature must be computed from the **post-interaction joint state**, with post-interaction marginals obtained by partial trace.

In [24]:
def kron_density(rho_a, rho_t):
    return torch.kron(rho_a, rho_t)

def partial_trace_over_text(rho_joint, rank):
    """
    Trace out the text subsystem.

    rho_joint ordering:
        [text, audio, text', audio']

    Returns:
        rho_audio [rank, rank]
    """

    rho = rho_joint.reshape(
        rank,
        rank,
        rank,
        rank,
    )

    # Trace over text dimension.
    # Diagonal of the text subsystem:
    rho_audio = torch.diagonal(
        rho,
        dim1=0,
        dim2=2,
    ).sum(dim=-1)

    return rho_audio


def partial_trace_over_audio(rho_joint, rank):
    """
    Trace out the audio subsystem.

    rho_joint ordering:
        [text, audio, text', audio']

    Returns:
        rho_text [rank, rank]
    """

    rho = rho_joint.reshape(
        rank,
        rank,
        rank,
        rank,
    )

    # Trace over audio dimension.
    rho_text = torch.diagonal(
        rho,
        dim1=1,
        dim2=3,
    ).sum(dim=-1)

    return rho_text

def quantum_mutual_information(rho_joint, rank):
    """
    Quantum mutual information:

        I(T:A) = S(rho_T') + S(rho_A') - S(rho_joint)

    where rho_T' and rho_A' are the post-interaction
    marginals obtained through partial trace.
    """

    rho_joint = symmetrize_density(rho_joint)

    rho_text = partial_trace_over_audio(
        rho_joint,
        rank,
    )

    rho_audio = partial_trace_over_text(
        rho_joint,
        rank,
    )

    rho_text = symmetrize_density(rho_text)
    rho_audio = symmetrize_density(rho_audio)

    joint_entropy = von_neumann_entropy(
        rho_joint
    )

    text_entropy = von_neumann_entropy(
        rho_text
    )

    audio_entropy = von_neumann_entropy(
        rho_audio
    )

    qmi = (
        text_entropy
        + audio_entropy
        - joint_entropy
    )

    return torch.clamp(
        qmi,
        min=0.0,
    )

print("Joint-state and partial-trace utilities defined.")


Joint-state and partial-trace utilities defined.


## Single-sample construction

This cell converts real hidden-state sequences into:

- reduced speech state \(\rho_a\)
- reduced text state \(\rho_t\)
- attention distributions
- purity
- entropy
- trace distance
- product joint state

In [19]:
def to_tensor_sequence(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2 or x.shape[1] != 768 or x.shape[0] < 1:
        raise ValueError(f"Expected [T, 768], got {x.shape}")
    return torch.from_numpy(x)

def build_modality_state(sequence, projector, density_head):
    x = to_tensor_sequence(sequence)
    with torch.no_grad():
        states = projector(x)
        rho, alpha = density_head(states)
    return rho, alpha, states

def build_quantum_representation(record, projector, density_head):
    akey = find_first_key(record, AUDIO_SEQUENCE_KEYS)
    tkey = find_first_key(record, TEXT_SEQUENCE_KEYS)

    if akey is None or tkey is None:
        raise ValueError("Record lacks both audio and text hidden-state sequences.")

    rho_a, alpha_a, _ = build_modality_state(record[akey], projector, density_head)
    rho_t, alpha_t, _ = build_modality_state(record[tkey], projector, density_head)

    rho_joint = kron_density(rho_a, rho_t)

    metrics = {
        "audio_trace": float(rho_a.trace()),
        "text_trace": float(rho_t.trace()),
        "joint_trace": float(rho_joint.trace()),
        "audio_purity": float(purity(rho_a)),
        "text_purity": float(purity(rho_t)),
        "audio_entropy": float(von_neumann_entropy(rho_a)),
        "text_entropy": float(von_neumann_entropy(rho_t)),
        "trace_distance": float(trace_distance(rho_a, rho_t)),
        "state_similarity": float(density_frobenius_similarity(rho_a, rho_t)),
        "joint_purity": float(purity(rho_joint)),
        "joint_entropy": float(von_neumann_entropy(rho_joint)),
        "product_joint_qmi": float(quantum_mutual_information(rho_joint, RANK)),
    }

    return {
        "rho_audio": rho_a.numpy(),
        "rho_text": rho_t.numpy(),
        "rho_joint_product": rho_joint.numpy(),
        "audio_attention": alpha_a.numpy(),
        "text_attention": alpha_t.numpy(),
        "metrics": metrics,
    }


## Sanity check: mixed-state non-degeneracy

For the corrected representation, inspect whether the density states actually become mixed.

Expected qualitative behavior:

- trace ≈ 1
- purity should generally be **below 1**
- entropy should generally be **above 0**
- values should not collapse to the previous rank-1 pure-state behavior

This is a **construct-validity check**, not a performance result.

In [20]:
def summarize_density(rho):
    rho = torch.as_tensor(rho, dtype=torch.float32)
    eigvals = torch.linalg.eigvalsh(symmetrize_density(rho)).detach().numpy()
    return {
        "trace": float(rho.trace()),
        "purity": float(purity(rho)),
        "entropy": float(von_neumann_entropy(rho)),
        "rank_numerical": int((eigvals > 1e-6).sum()),
        "min_eigenvalue": float(eigvals.min()),
        "max_eigenvalue": float(eigvals.max()),
    }

print("The next cell requires at least one record with real hidden-state sequences.")


The next cell requires at least one record with real hidden-state sequences.


## Save the corrected representation

Before processing the full dataset, validate a small number of records and inspect the resulting metrics. Then freeze the projection/attention parameters and use the same fitted representation consistently for train/dev/test.

In [ ]:
# Build one real representation as a sanity check.
example = train_records[0]

result = build_quantum_representation(
    example,
    projector,
    density_head,
)

print("Participant:", example["participant_id"])
print("Utterance  :", example["utterance_id"])

print("\nMetrics:")
for key, value in result["metrics"].items():
    print(f"  {key}: {value:.8f}")

print("\nAudio density:")
print(summarize_density(result["rho_audio"]))

print("\nText density:")
print(summarize_density(result["rho_text"]))

# Basic density-state checks

assert abs(result["metrics"]["audio_trace"] - 1.0) < 1e-5, \
    "Audio density is not unit trace."

assert abs(result["metrics"]["text_trace"] - 1.0) < 1e-5, \
    "Text density is not unit trace."

assert abs(result["metrics"]["joint_trace"] - 1.0) < 1e-5, \
    "Joint density is not unit trace."

# The initial tensor-product state is uncorrelated,
# so its QMI should be approximately zero.
assert abs(result["metrics"]["product_joint_qmi"]) < 1e-5, \
    "Product-state QMI should be approximately zero."

print("\nBasic density-state sanity checks: PASS")

print(
    "NOTE: product_joint_qmi ~= 0 is expected for the "
    "uncoupled product state. Post-interaction QMI should "
    "be computed from the reduced post-interaction marginals."
)

Checkpoint:

1. The notebook consumes `data/processed/quantum/train.pkl`, `dev.pkl`, and `test.pkl`.
2. Participant counts are exactly **163 train / 56 dev / 56 test**, with no overlap.
3. Audio and text inputs are real hidden-state sequences `[T, 768]`.
4. Each modality becomes an attention-weighted mixed density state with reduced rank `r=16`.
5. The joint product state has dimension `r² = 256`.
6. The product-state QMI is expected to be approximately zero because the state is explicitly `rho_a ⊗ rho_t`.
7. The final proposed interaction QMI is **not** claimed here; it must be computed from a post-interaction/Kraus joint state and post-interaction partial-trace marginals.

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SpeechConditionedKraus(nn.Module):
    """
    Speech-conditioned Kraus channel.

    Input:
        rho_text  : [r, r] text density matrix
        rho_audio : [r, r] audio density matrix

    Output:
        rho_text_post : [r, r] post-interaction text density matrix

    The channel is:
        rho'_t = sum_k K_k rho_t K_k^\dagger

    with:
        sum_k K_k^\dagger K_k = I

    The Kraus operators are generated from the audio state.
    """

    def __init__(self, rank=16, num_kraus=4, strength=0.25):
        super().__init__()

        self.rank = rank
        self.num_kraus = num_kraus
        self.strength = strength

        # Audio-state descriptor -> parameters for Kraus operators
        self.conditioner = nn.Sequential(
            nn.Linear(rank * rank, 128),
            nn.GELU(),
            nn.Linear(128, num_kraus * rank * rank),
        )

    def forward(self, rho_text, rho_audio):

        r = self.rank

        # ---------------------------------------------------------
        # Flatten audio density state
        # ---------------------------------------------------------
        audio_flat = rho_audio.reshape(-1)

        # ---------------------------------------------------------
        # Generate raw Kraus operators
        # ---------------------------------------------------------
        raw = self.conditioner(audio_flat)

        raw = raw.reshape(
            self.num_kraus,
            r,
            r
        )

        # ---------------------------------------------------------
        # Convert raw operators into positive channel weights
        # ---------------------------------------------------------
        # We use A_k^T A_k as positive operators and normalize
        # their sum so that the channel is trace preserving.
        positive_ops = []

        for k in range(self.num_kraus):
            A = raw[k]

            # Stabilise the operator
            A = self.strength * A

            positive_ops.append(A)

        # ---------------------------------------------------------
        # Build completeness operator
        #
        # C = sum_k A_k^T A_k
        # ---------------------------------------------------------
        C = torch.zeros(
            r,
            r,
            device=rho_text.device,
            dtype=rho_text.dtype,
        )

        for A in positive_ops:
            C = C + A.T @ A

        # Add identity stabilisation
        eps = 1e-6
        C = C + eps * torch.eye(
            r,
            device=rho_text.device,
            dtype=rho_text.dtype,
        )

        # ---------------------------------------------------------
        # C^{-1/2}
        # ---------------------------------------------------------
        eigenvalues, eigenvectors = torch.linalg.eigh(C)

        eigenvalues = torch.clamp(
            eigenvalues,
            min=eps
        )

        C_inv_sqrt = (
            eigenvectors
            @ torch.diag(eigenvalues.rsqrt())
            @ eigenvectors.T
        )

        # ---------------------------------------------------------
        # Normalize each Kraus operator
        #
        # K_k = A_k C^{-1/2}
        #
        # Therefore:
        #
        # sum K_k^T K_k = I
        # ---------------------------------------------------------
        kraus_ops = []

        for A in positive_ops:
            K = A @ C_inv_sqrt
            kraus_ops.append(K)

        # ---------------------------------------------------------
        # Apply Kraus channel
        #
        # rho'_t = sum K rho_t K^T
        # ---------------------------------------------------------
        rho_post = torch.zeros_like(rho_text)

        for K in kraus_ops:
            rho_post = (
                rho_post
                + K @ rho_text @ K.T
            )

        # ---------------------------------------------------------
        # Numerical cleanup
        # ---------------------------------------------------------
        rho_post = 0.5 * (
            rho_post
            + rho_post.T
        )

        # Trace normalization
        rho_post = (
            rho_post
            / torch.trace(rho_post).clamp_min(eps)
        )

        return rho_post, kraus_ops

In [27]:
RANK = 16
NUM_KRAUS = 4

kraus_channel = SpeechConditionedKraus(
    rank=RANK,
    num_kraus=NUM_KRAUS,
    strength=0.25,
)

kraus_channel.eval()

SpeechConditionedKraus(
  (conditioner): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=128, out_features=1024, bias=True)
  )
)

In [30]:
# Convert density matrices to PyTorch tensors
device = next(kraus_channel.parameters()).device

rho_text = torch.as_tensor(
    result["rho_text"],
    dtype=torch.float32,
    device=device,
)

rho_audio = torch.as_tensor(
    result["rho_audio"],
    dtype=torch.float32,
    device=device,
)

# Apply speech-conditioned Kraus channel
rho_text_post, kraus_ops = kraus_channel(
    rho_text,
    rho_audio,
)

print("Before Kraus interaction:")
print(summarize_density(rho_text))

print("\nAfter Kraus interaction:")
print(summarize_density(rho_text_post))

Before Kraus interaction:
{'trace': 1.0, 'purity': 0.26713502407073975, 'entropy': 1.5032627582550049, 'rank_numerical': 8, 'min_eigenvalue': -1.0116772308776945e-08, 'max_eigenvalue': 0.35739651322364807}

After Kraus interaction:
{'trace': 1.0, 'purity': 0.12714169919490814, 'entropy': 2.302206039428711, 'rank_numerical': 16, 'min_eigenvalue': 0.0006869416683912277, 'max_eigenvalue': 0.2240608185529709}


In [31]:
print("rho_text:")
print("  type  :", type(result["rho_text"]))
print("  shape :", np.asarray(result["rho_text"]).shape)

print("\nrho_audio:")
print("  type  :", type(result["rho_audio"]))
print("  shape :", np.asarray(result["rho_audio"]).shape)

print("\nConverted:")
print("  rho_text :", rho_text.dtype, rho_text.shape)
print("  rho_audio:", rho_audio.dtype, rho_audio.shape)

rho_text:
  type  : <class 'numpy.ndarray'>
  shape : (16, 16)

rho_audio:
  type  : <class 'numpy.ndarray'>
  shape : (16, 16)

Converted:
  rho_text : torch.float32 torch.Size([16, 16])
  rho_audio: torch.float32 torch.Size([16, 16])


In [32]:
rho_joint_product = torch.kron(
    rho_text,
    rho_audio,
)

In [33]:
print(
    "Product joint trace:",
    torch.trace(rho_joint_product).item()
)

Product joint trace: 1.0


In [34]:
def apply_kraus_to_joint(
    rho_joint,
    kraus_ops,
    rank,
):
    """
    Apply speech-conditioned Kraus operators to the
    text subsystem of a bipartite joint state.

    rho_joint shape:
        [rank^2, rank^2]

    K acts on the text subsystem:

        L_k = K_k ⊗ I_audio

        rho' = sum_k L_k rho L_k^T
    """

    I_audio = torch.eye(
        rank,
        dtype=rho_joint.dtype,
        device=rho_joint.device,
    )

    rho_post = torch.zeros_like(rho_joint)

    for K in kraus_ops:

        L = torch.kron(
            K,
            I_audio
        )

        rho_post += (
            L
            @ rho_joint
            @ L.T
        )

    # Symmetrize
    rho_post = 0.5 * (
        rho_post
        + rho_post.T
    )

    # Normalize
    eps = 1e-8

    rho_post = (
        rho_post
        / torch.trace(rho_post).clamp_min(eps)
    )

    return rho_post

In [35]:
rho_joint_post = apply_kraus_to_joint(
    rho_joint_product,
    kraus_ops,
    RANK,
)

print(
    "Post-interaction joint trace:",
    torch.trace(rho_joint_post).item()
)

assert abs(
    torch.trace(rho_joint_post).item() - 1.0
) < 1e-5

print("Post-interaction joint state: PASS")

Post-interaction joint trace: 1.0
Post-interaction joint state: PASS


In [36]:
rho_audio_post = partial_trace_over_text(
    rho_joint_post,
    RANK,
)

rho_text_post_joint = partial_trace_over_audio(
    rho_joint_post,
    RANK,
)

In [37]:
print("Post-interaction audio:")
print(summarize_density(rho_audio_post))

print("\nPost-interaction text:")
print(summarize_density(rho_text_post_joint))

Post-interaction audio:
{'trace': 1.0, 'purity': 0.14869101345539093, 'entropy': 2.2147037982940674, 'rank_numerical': 16, 'min_eigenvalue': 0.0030020466074347496, 'max_eigenvalue': 0.27181267738342285}

Post-interaction text:
{'trace': 1.0, 'purity': 0.12714169919490814, 'entropy': 2.3022055625915527, 'rank_numerical': 16, 'min_eigenvalue': 0.0006869444623589516, 'max_eigenvalue': 0.2240607738494873}


In [38]:
assert abs(
    torch.trace(rho_audio_post).item() - 1.0
) < 1e-5

assert abs(
    torch.trace(rho_text_post_joint).item() - 1.0
) < 1e-5

print("\nPost-interaction marginal traces: PASS")


Post-interaction marginal traces: PASS


In [39]:
def post_interaction_qmi(
    rho_joint,
    rank,
):
    rho_audio = partial_trace_over_text(
        rho_joint,
        rank,
    )

    rho_text = partial_trace_over_audio(
        rho_joint,
        rank,
    )

    rho_audio = symmetrize_density(
        rho_audio
    )

    rho_text = symmetrize_density(
        rho_text
    )

    rho_joint = symmetrize_density(
        rho_joint
    )

    S_audio = von_neumann_entropy(
        rho_audio
    )

    S_text = von_neumann_entropy(
        rho_text
    )

    S_joint = von_neumann_entropy(
        rho_joint
    )

    qmi = (
        S_audio
        + S_text
        - S_joint
    )

    return torch.clamp(
        qmi,
        min=0.0
    )

In [40]:
qmi_before = quantum_mutual_information(
    rho_joint_product,
    RANK,
)

qmi_after = post_interaction_qmi(
    rho_joint_post,
    RANK,
)

print("QMI before interaction :", qmi_before.item())
print("QMI after interaction  :", qmi_after.item())

QMI before interaction : 0.0
QMI after interaction  : 0.0
